# Annotate Dave protocol with bit information

Inserts per-round XML comments into the Dave recipe generated by notebook 04, naming which bits/colors are imaged in each round -- so the recipe is self-documenting when read/edited later (in Dave itself or by hand). Shared by every pipeline/backend, since it only needs the Dave XML (notebook 04) and the round-bit-color mapping (notebook 03) -- neither MERlin's data organization nor fishtank's color usage. Used to live inside `06_create_data_organization.ipynb`, but that notebook doesn't exist for the fishtank backend (`06_create_color_usage.ipynb` instead), so this step is its own notebook here rather than duplicated into both.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT027_saving_time/
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR = SAMPLE_DIR / "settings"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.dave import annotate_dave_with_round_info, dave_config_filename
from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.pipeline_config import load_pipeline_config

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale.
PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

MICROSCOPE  = PIPELINE_CONFIG.microscope
# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 01-04 used (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")

## Round – bit – color mapping

The round → bit → colour mapping is defined in **notebook 03** (which derives
`N_HYBS` from it) and saved to `round_bit_color_map.csv`. Here it is read back for
the Dave annotation. Re-run notebook 03 to change the mapping.

In [ ]:
# round_bit_color is defined & saved in notebook 03; read it back here.
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(
        f"{rbc_path} not found — run notebook 03 first "
        f"(it now defines the round–bit–color mapping and N_HYBS)."
    )
rbc_df          = pd.read_csv(rbc_path)
round_bit_color = [(int(r), int(b), int(c))
                   for r, b, c in rbc_df[["round", "bit", "color"]].itertuples(index=False, name=None)]
print(f"Loaded {len(round_bit_color)} (round, bit, color) rows from {rbc_path.name}")
print(rbc_df.to_string(index=False))

## Annotate Dave XML with bit information

Adds per-round XML comments to the Dave config generated by notebook 04.
Imaging round 1 is the cells acquisition (no bits), so the bit comments attach to
the fluidics loops that precede each bits imaging round (rounds 2…N+1). The
hyb/bit-indexed `round_bit_color` is shifted `+1` here to match those imaging-round
numbers.

Re-run this cell whenever the round–bit–color mapping changes.

In [ ]:
# Construct the exact expected filename (MICROSCOPE + N_HYBS, from notebook 03's
# round_bit_color_map.csv, read above) rather than globbing settings/dave-*.xml
# and guessing which match is "the" recipe -- that guess breaks as soon as more
# than one dave-*.xml lives in this settings/ folder (e.g. two acquisitions
# sharing one sample folder), since alphabetical sort picks "…-13hybs-…" before
# "…-9hybs-…" (string comparison, not numeric).
N_HYBS    = int(rbc_df["round"].max())
dave_path = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

if not dave_path.exists():
    print(f"No {dave_path.name} found in settings/ — run notebook 04 first.")
else:
    print(f"Annotating: {dave_path.name}")
    # round_bit_color is hyb/bit-indexed (1..N) for data-organization, but the
    # Dave recipe images bits in imaging rounds 2..N+1 (round 1 = cells), so the
    # annotation round indices are shifted +1 to line up with the Fluidics loops.
    annotate_rbc = [(r + 1, bit, color) for (r, bit, color) in round_bit_color]
    annotate_dave_with_round_info(dave_path, annotate_rbc)
    print("Done. Preview of annotated file:")
    with open(dave_path, encoding="ISO-8859-1") as fh:
        print(fh.read())